# Dashcam Video Collision Prediction

This notebook uses the pretrained **MViT V2-S** model to estimate the probability of a collision in dashcam videos. For transfer learning, only the final Transformer block, the normalization layer, and the classification head are trainable.

The data is loaded from the Nexar competition dataset on Kaggle. Attach the competition dataset to the notebook, enable a GPU accelerator, and enable Internet access if the pretrained weights need to be downloaded.


## 1. Environment Setup

Import the required libraries, define the main configuration values and data paths, and select the computation device.


In [11]:
import gc
import random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torchvision.models.video import MViT_V2_S_Weights, mvit_v2_s
from tqdm.auto import tqdm

SEED = 42
NUM_FRAMES = 16
IMAGE_SIZE = 224
CLIP_DURATION = 10.0
HORIZONS = (0.5, 1.0, 1.5)
BATCH_SIZE = 2
NUM_EPOCHS = 3

DATA_ROOT = Path("/kaggle/input/competitions/nexar-collision-prediction")
WORK_ROOT = Path("/kaggle/working")
CHECKPOINT_PATH = WORK_ROOT / "best_mvit_v2_s.pth"

if not DATA_ROOT.exists():
    raise FileNotFoundError("Competition dataset was not found.")

WORK_ROOT.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
cv2.setNumThreads(0)
torch.backends.cudnn.benchmark = True

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

if device.type != "cuda":
    raise RuntimeError("Please enable a GPU accelerator in Kaggle settings.")

print("PyTorch:", torch.__version__)
print("Device:", device)
print("GPU count:", torch.cuda.device_count())

for index in range(torch.cuda.device_count()):
    print(f"GPU {index}:", torch.cuda.get_device_name(index))


PyTorch: 2.11.0+cu130
Device: cuda:0
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


## 2. Load Competition Data

Keep video IDs as strings so that leading zeros are preserved when reading the competition CSV files.


In [12]:
def normalize_id(value):
    value = str(value).strip()
    if value.endswith(".0"):
        value = value[:-2]
    return value.zfill(5)


train_df = pd.read_csv(DATA_ROOT / "train.csv", dtype={"id": str})
test_df = pd.read_csv(DATA_ROOT / "test.csv", dtype={"id": str})
sample_submission = pd.read_csv(
    DATA_ROOT / "sample_submission.csv",
    dtype={"id": str}
)

for frame in [train_df, test_df, sample_submission]:
    frame["id"] = frame["id"].map(normalize_id)

train_df["target"] = train_df["target"].astype(int)

assert train_df["id"].is_unique
assert test_df["id"].is_unique
assert sample_submission["id"].is_unique
assert train_df["target"].isin([0, 1]).all()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Target distribution:")
print(train_df["target"].value_counts().sort_index())
print("Submission columns:", sample_submission.columns.tolist())


Train shape: (1500, 4)
Test shape: (1344, 1)
Target distribution:
target
0    750
1    750
Name: count, dtype: int64
Submission columns: ['id', 'target']


## 3. Extract Video Metadata

Calculate each video's duration from its frame rate and frame count. Extract this metadata once and save it in the working directory for reuse when building temporal clips.


In [13]:
def collect_video_metadata(split, video_ids):
    video_dir = DATA_ROOT / split
    records = []

    for video_id in tqdm(video_ids, desc=f"Scanning {split} videos"):
        video_path = video_dir / f"{video_id}.mp4"
        cap = cv2.VideoCapture(str(video_path))

        if not cap.isOpened():
            cap.release()
            raise RuntimeError(f"Could not open video: {video_path}")

        fps = float(cap.get(cv2.CAP_PROP_FPS))
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.release()

        if fps <= 0 or frame_count <= 0:
            raise RuntimeError(f"Invalid video metadata: {video_path}")

        records.append({
            "id": video_id,
            "fps": fps,
            "frame_count": frame_count,
            "duration_sec": frame_count / fps
        })

    return pd.DataFrame(records)


train_video_meta = collect_video_metadata("train", train_df["id"].tolist())
test_video_meta = collect_video_metadata("test", test_df["id"].tolist())

train_data = train_df.merge(
    train_video_meta,
    on="id",
    how="left",
    validate="one_to_one"
)

test_data = test_df.merge(
    test_video_meta,
    on="id",
    how="left",
    validate="one_to_one"
)

assert train_data["duration_sec"].notna().all()
assert test_data["duration_sec"].notna().all()
assert len(train_data) == 1500
assert len(test_data) == 1344

train_video_meta.to_csv(WORK_ROOT / "train_video_metadata.csv", index=False)
test_video_meta.to_csv(WORK_ROOT / "test_video_metadata.csv", index=False)

print("Train videos:", len(train_data))
print("Test videos:", len(test_data))
print("Median train duration:", round(train_data["duration_sec"].median(), 3))
print("Median test duration:", round(test_data["duration_sec"].median(), 3))


Scanning train videos:   0%|          | 0/1500 [00:00<?, ?it/s]

Scanning test videos:   0%|          | 0/1344 [00:00<?, ?it/s]

Train videos: 1500
Test videos: 1344
Median train duration: 40.131
Median test duration: 9.96


## 4. Split Data and Build Temporal Clips

Split the videos at the video level using stratification. All clips from one video remain in the same split to prevent video-level leakage.

The positive training fold is used to estimate the alert lead-time distribution and the event-time offset distribution. These training-only reference statistics are then used to construct negative temporal samples in both training and validation.

For positive videos, create a clip ending at each eligible prediction horizon before the annotated collision event. For negative videos, sample a synthetic event time and alert lead time from the training reference distributions, then create negative clips using the same temporal eligibility rules.

This negative-sample construction is an approximation because negative videos do not have annotated collision or alert timestamps. The reference distributions are estimated from the training fold only.


In [14]:
train_fold, val_fold = train_test_split(
    train_data,
    test_size=0.2,
    random_state=SEED,
    stratify=train_data["target"]
)

train_fold = train_fold.reset_index(drop=True)
val_fold = val_fold.reset_index(drop=True)

assert set(train_fold["id"]).isdisjoint(set(val_fold["id"]))

positive_reference = train_fold[train_fold["target"] == 1].copy()

positive_leads = (
    positive_reference["time_of_event"] -
    positive_reference["time_of_alert"]
).dropna().to_numpy()

event_offsets = (
    positive_reference["time_of_event"] -
    positive_reference["duration_sec"] / 2
).dropna().to_numpy()

event_offset_mean = float(np.mean(event_offsets))
event_offset_std = float(np.std(event_offsets, ddof=1))

assert len(positive_leads) > 0
assert event_offset_std > 0


def build_clip_index(frame, seed):
    rng = np.random.default_rng(seed)
    records = []
    skipped = 0

    for row in frame.itertuples(index=False):
        video_id = normalize_id(row.id)
        duration = float(row.duration_sec)
        video_path = DATA_ROOT / "train" / f"{video_id}.mp4"

        if duration <= 0:
            skipped += 1
            continue

        if int(row.target) == 1:
            event_time = float(row.time_of_event)
            alert_time = float(row.time_of_alert)
            lead_time = event_time - alert_time

            for horizon in HORIZONS:
                if horizon > 0.5 and lead_time <= horizon:
                    continue

                clip_end = event_time - horizon

                if clip_end <= 0 or clip_end > duration:
                    skipped += 1
                    continue

                clip_start = max(0.0, clip_end - CLIP_DURATION)

                records.append({
                    "id": video_id,
                    "video_path": str(video_path),
                    "clip_start": clip_start,
                    "clip_end": clip_end,
                    "clip_duration": clip_end - clip_start,
                    "horizon": float(horizon),
                    "label": 1,
                    "video_target": 1,
                    "duration_sec": duration
                })

        else:
            fake_event_time = (
                duration / 2 +
                rng.normal(event_offset_mean, event_offset_std)
            )

            fake_lead_time = float(rng.choice(positive_leads))

            for horizon in HORIZONS:
                if horizon > 0.5 and fake_lead_time <= horizon:
                    continue

                clip_end = fake_event_time - horizon

                if clip_end <= 0 or clip_end > duration:
                    skipped += 1
                    continue

                clip_start = max(0.0, clip_end - CLIP_DURATION)

                records.append({
                    "id": video_id,
                    "video_path": str(video_path),
                    "clip_start": clip_start,
                    "clip_end": clip_end,
                    "clip_duration": clip_end - clip_start,
                    "horizon": float(horizon),
                    "label": 0,
                    "video_target": 0,
                    "duration_sec": duration
                })

    clips = pd.DataFrame(records)

    if clips.empty:
        raise RuntimeError("No temporal clips were generated.")

    print("Skipped invalid clips:", skipped)

    return clips


train_clips = build_clip_index(train_fold, seed=42)
val_clips = build_clip_index(val_fold, seed=43)

for split_name, clips in [
    ("TRAIN", train_clips),
    ("VALIDATION", val_clips)
]:
    print("\n" + "=" * 55)
    print(split_name)
    print("=" * 55)
    print("Total clips:", len(clips))
    print("Unique videos:", clips["id"].nunique())
    print("\nLabel distribution by horizon:")
    print(
        clips.groupby(["horizon", "label"])
        .size()
        .unstack(fill_value=0)
    )

    assert clips["label"].isin([0, 1]).all()
    assert (clips["label"] == clips["video_target"]).all()
    assert not clips.duplicated(["id", "horizon"]).any()
    assert clips["video_path"].map(lambda value: Path(value).is_file()).all()

assert set(train_clips["id"]).isdisjoint(set(val_clips["id"]))

train_clips.to_csv(WORK_ROOT / "train_clips.csv", index=False)
val_clips.to_csv(WORK_ROOT / "val_clips.csv", index=False)

print("\nTEMPORAL DATASET CONSTRUCTION COMPLETED.")


Skipped invalid clips: 0
Skipped invalid clips: 0

TRAIN
Total clips: 2667
Unique videos: 1200

Label distribution by horizon:
label      0    1
horizon          
0.5      600  600
1.0      454  452
1.5      281  280

VALIDATION
Total clips: 657
Unique videos: 300

Label distribution by horizon:
label      0    1
horizon          
0.5      150  150
1.0      110  104
1.5       76   67

TEMPORAL DATASET CONSTRUCTION COMPLETED.


## 5. Build Dataset and DataLoaders

Uniformly sample 16 frames from each clip. Convert the frames to RGB, resize them to 224 × 224 pixels, and normalize them using the Kinetics dataset statistics.


In [15]:
MEAN = np.array(
    [0.43216, 0.394666, 0.37645],
    dtype=np.float32
).reshape(1, 1, 1, 3)

STD = np.array(
    [0.22803, 0.22145, 0.216989],
    dtype=np.float32
).reshape(1, 1, 1, 3)


class DashcamClipDataset(Dataset):
    def __init__(self, clips, num_frames=NUM_FRAMES, image_size=IMAGE_SIZE):
        self.clips = clips.reset_index(drop=True).copy()
        self.num_frames = num_frames
        self.image_size = image_size

    def __len__(self):
        return len(self.clips)

    def __getitem__(self, index):
        row = self.clips.iloc[index]
        video_path = Path(row["video_path"])
        cap = cv2.VideoCapture(str(video_path))

        if not cap.isOpened():
            cap.release()
            raise RuntimeError(f"Could not open video: {video_path}")

        try:
            fps = float(cap.get(cv2.CAP_PROP_FPS))
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

            if fps <= 0 or total_frames <= 0:
                raise RuntimeError(f"Invalid video metadata: {video_path}")

            start_frame = int(np.floor(float(row["clip_start"]) * fps))
            end_frame = int(np.floor(float(row["clip_end"]) * fps))
            start_frame = max(0, min(start_frame, total_frames - 1))
            end_frame = max(start_frame, min(end_frame, total_frames - 1))

            frame_indices = np.linspace(
                start_frame,
                end_frame,
                self.num_frames
            ).round().astype(np.int64)

            positions = {}
            for position, frame_index in enumerate(frame_indices):
                positions.setdefault(int(frame_index), []).append(position)

            frames = [None] * self.num_frames
            cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

            for frame_index in range(start_frame, end_frame + 1):
                success, frame = cap.read()

                if not success:
                    break

                if frame_index in positions:
                    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    frame = cv2.resize(
                        frame,
                        (self.image_size, self.image_size),
                        interpolation=cv2.INTER_AREA
                    )

                    for position in positions[frame_index]:
                        frames[position] = frame

            available = [
                (position, frame)
                for position, frame in enumerate(frames)
                if frame is not None
            ]

            if not available:
                raise RuntimeError(f"No frames could be decoded: {video_path}")

            valid_positions = [position for position, _ in available]

            for position in range(self.num_frames):
                if frames[position] is None:
                    nearest = min(
                        valid_positions,
                        key=lambda valid_position: abs(valid_position - position)
                    )
                    frames[position] = frames[nearest]
        finally:
            cap.release()

        frames = np.stack(frames).astype(np.float32) / 255.0
        frames = (frames - MEAN) / STD
        frames = np.transpose(frames, (3, 0, 1, 2)).copy()

        video_tensor = torch.from_numpy(frames)
        label = torch.tensor(float(row["label"]), dtype=torch.float32)
        horizon = torch.tensor(float(row["horizon"]), dtype=torch.float32)

        return video_tensor, label, str(row["id"]), horizon


def make_loader(clips, shuffle):
    dataset = DashcamClipDataset(clips)

    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=True,
        drop_last=False
    )


train_loader = make_loader(train_clips, True)
val_loader = make_loader(val_clips, False)

sample_videos, sample_labels, sample_ids, sample_horizons = next(iter(train_loader))

print("Video batch shape:", tuple(sample_videos.shape))
print("Label shape:", tuple(sample_labels.shape))
print("Sample IDs:", sample_ids[:3])
print("Sample horizons:", sample_horizons[:3].tolist())

assert sample_videos.shape == (BATCH_SIZE, 3, NUM_FRAMES, IMAGE_SIZE, IMAGE_SIZE)
assert torch.isfinite(sample_videos).all()

del sample_videos, sample_labels, sample_ids, sample_horizons


Video batch shape: (2, 3, 16, 224, 224)
Label shape: (2,)
Sample IDs: ('00671', '01697')
Sample horizons: [1.0, 1.0]


## 6. Load and Prepare the MViT V2-S Model

Load the pretrained weights from Kinetics-400. Replace the classification head with a two-class output layer. For fine-tuning, unfreeze only the final Transformer block, the normalization layer, and the classification head.


In [16]:
weights = MViT_V2_S_Weights.DEFAULT
model = mvit_v2_s(weights=weights)
model.head[1] = nn.Linear(model.head[1].in_features, 2)

for parameter in model.parameters():
    parameter.requires_grad = False

for parameter in model.blocks[-1].parameters():
    parameter.requires_grad = True

for parameter in model.norm.parameters():
    parameter.requires_grad = True

for parameter in model.head.parameters():
    parameter.requires_grad = True

if torch.cuda.device_count() > 1:
    model = nn.DataParallel(
        model,
        device_ids=list(range(torch.cuda.device_count()))
    )

model = model.to(device)
base_model = model.module if isinstance(model, nn.DataParallel) else model

block_params = list(base_model.blocks[-1].parameters())
norm_params = list(base_model.norm.parameters())
head_params = list(base_model.head.parameters())

optimizer = torch.optim.AdamW(
    [
        {"params": block_params, "lr": 1e-5},
        {"params": norm_params, "lr": 1e-5},
        {"params": head_params, "lr": 1e-4}
    ],
    weight_decay=1e-4
)

criterion = nn.CrossEntropyLoss()

total_params = sum(parameter.numel() for parameter in base_model.parameters())
trainable_params = sum(
    parameter.numel()
    for parameter in base_model.parameters()
    if parameter.requires_grad
)

print("Architecture:", type(base_model).__name__)
print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")
print("Frozen parameters:", f"{total_params - trainable_params:,}")
print("Optimizer:", type(optimizer).__name__)
print("Block learning rate:", optimizer.param_groups[0]["lr"])
print("Head learning rate:", optimizer.param_groups[2]["lr"])


Architecture: MViT
Total parameters: 34,231,682
Trainable parameters: 7,103,234
Frozen parameters: 27,128,448
Optimizer: AdamW
Block learning rate: 1e-05
Head learning rate: 0.0001


## 7. Model Training and Evaluation

At each epoch, update the model using the training data and then evaluate it on the validation data. The main evaluation metric is the mean Average Precision across the three time horizons. Save the checkpoint that achieves the best validation score.


In [17]:
def evaluate_model(model, loader):
    model.eval()

    all_labels = []
    all_probabilities = []
    all_horizons = []
    total_loss = 0.0
    total_samples = 0

    with torch.inference_mode():
        for videos, labels, video_ids, horizons in tqdm(
            loader,
            desc="Validation",
            leave=False
        ):
            videos = videos.to(device, non_blocking=True)
            labels = labels.to(device).long().view(-1)

            logits = model(videos)
            loss = criterion(logits, labels)
            probabilities = torch.softmax(logits.float(), dim=1)[:, 1]

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size

            all_labels.extend(labels.cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())
            all_horizons.extend(horizons.cpu().numpy())

    y_true = np.asarray(all_labels).astype(int)
    y_prob = np.asarray(all_probabilities).astype(float)
    horizon_values = np.asarray(all_horizons).astype(float)

    assert len(y_true) == len(loader.dataset)
    assert np.isfinite(y_prob).all()

    results = []

    for horizon in HORIZONS:
        mask = np.isclose(horizon_values, horizon)
        labels_h = y_true[mask]
        probabilities_h = y_prob[mask]

        if len(np.unique(labels_h)) < 2:
            raise ValueError(f"Both classes are required for horizon {horizon}.")

        results.append({
            "Horizon": horizon,
            "AP": average_precision_score(labels_h, probabilities_h),
            "ROC-AUC": roc_auc_score(labels_h, probabilities_h),
            "Baseline AP": labels_h.mean()
        })

    results_df = pd.DataFrame(results)
    mean_ap = float(results_df["AP"].mean())
    average_loss = total_loss / total_samples

    return results_df, mean_ap, average_loss


history = []
best_val_map = -1.0

for epoch in range(NUM_EPOCHS):
    model.train()
    running_train_loss = 0.0
    train_samples = 0

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{NUM_EPOCHS} - Training"
    )

    for videos, labels, video_ids, horizons in progress:
        videos = videos.to(device, non_blocking=True)
        labels = labels.to(device).long().view(-1)

        optimizer.zero_grad(set_to_none=True)
        logits = model(videos)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        running_train_loss += loss.item() * batch_size
        train_samples += batch_size
        progress.set_postfix(loss=f"{loss.item():.4f}")

    train_loss = running_train_loss / train_samples
    val_results, val_map, val_loss = evaluate_model(model, val_loader)

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_map": val_map
    })

    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")
    print(f"Training loss: {train_loss:.4f}")
    print(f"Validation loss: {val_loss:.4f}")
    print(val_results.round(4).to_string(index=False))
    print(f"Mean AP: {val_map:.4f}")

    if np.isfinite(val_map) and val_map > best_val_map:
        best_val_map = val_map
        torch.save(base_model.state_dict(), CHECKPOINT_PATH)
        print("Best checkpoint saved.")

    gc.collect()
    torch.cuda.empty_cache()

history_df = pd.DataFrame(history)
history_df.to_csv(WORK_ROOT / "mvit_training_history.csv", index=False)

if not CHECKPOINT_PATH.exists():
    raise RuntimeError("Training did not produce a valid checkpoint.")

print("\nTraining completed.")
print("Best validation mAP:", round(best_val_map, 4))


Epoch 1/3 - Training:   0%|          | 0/1334 [00:00<?, ?it/s]

Validation:   0%|          | 0/329 [00:00<?, ?it/s]


Epoch 1/3
Training loss: 0.6758
Validation loss: 0.6561
 Horizon     AP  ROC-AUC  Baseline AP
     0.5 0.7040   0.7086       0.5000
     1.0 0.6787   0.7071       0.4860
     1.5 0.6399   0.6972       0.4685
Mean AP: 0.6742
Best checkpoint saved.


Epoch 2/3 - Training:   0%|          | 0/1334 [00:00<?, ?it/s]

Validation:   0%|          | 0/329 [00:00<?, ?it/s]


Epoch 2/3
Training loss: 0.6360
Validation loss: 0.6276
 Horizon     AP  ROC-AUC  Baseline AP
     0.5 0.7115   0.7147       0.5000
     1.0 0.6894   0.7151       0.4860
     1.5 0.6472   0.7074       0.4685
Mean AP: 0.6827
Best checkpoint saved.


Epoch 3/3 - Training:   0%|          | 0/1334 [00:00<?, ?it/s]

Validation:   0%|          | 0/329 [00:00<?, ?it/s]


Epoch 3/3
Training loss: 0.6059
Validation loss: 0.6256
 Horizon     AP  ROC-AUC  Baseline AP
     0.5 0.7232   0.7180       0.5000
     1.0 0.6911   0.7182       0.4860
     1.5 0.6737   0.7256       0.4685
Mean AP: 0.6960
Best checkpoint saved.

Training completed.
Best validation mAP: 0.696


## 8. Restore the Best Checkpoint and Run Final Evaluation

After training, load the best checkpoint and calculate the validation metrics again to confirm the selected model's performance.


In [18]:
state_dict = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=True
)

base_model.load_state_dict(state_dict, strict=True)
model.eval()

val_results, final_val_map, final_val_loss = evaluate_model(model, val_loader)

print("Best checkpoint:", CHECKPOINT_PATH)
print("Validation loss:", round(final_val_loss, 4))
print("Validation mAP:", round(final_val_map, 4))
print("\nHorizon results:")
print(val_results.round(4).to_string(index=False))


Validation:   0%|          | 0/329 [00:00<?, ?it/s]

Best checkpoint: /kaggle/working/best_mvit_v2_s.pth
Validation loss: 0.6256
Validation mAP: 0.696

Horizon results:
 Horizon     AP  ROC-AUC  Baseline AP
     0.5 0.7232   0.7180       0.5000
     1.0 0.6911   0.7182       0.4860
     1.5 0.6737   0.7256       0.4685


## 9. Build Test Set Clips

For each test video, create three clips of up to 10 seconds from different positions in the video. Load these clips using the same dataset implementation and preprocessing settings used for training and validation.


In [19]:
NUM_TEST_CLIPS = 3

test_records = []

for row in test_data.itertuples(index=False):
    video_id = normalize_id(row.id)
    duration = float(row.duration_sec)
    clip_length = min(CLIP_DURATION, duration)
    max_start = max(0.0, duration - clip_length)
    starts = np.linspace(0.0, max_start, num=NUM_TEST_CLIPS)
    video_path = DATA_ROOT / "test" / f"{video_id}.mp4"

    for clip_start in starts:
        clip_end = float(clip_start + clip_length)

        test_records.append({
            "id": video_id,
            "video_path": str(video_path),
            "clip_start": float(clip_start),
            "clip_end": clip_end,
            "clip_duration": clip_end - float(clip_start),
            "horizon": 1.0,
            "label": 0,
            "video_target": 0,
            "duration_sec": duration
        })

test_clips = pd.DataFrame(test_records)
test_dataset = DashcamClipDataset(test_clips)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

assert len(test_clips) == len(test_data) * NUM_TEST_CLIPS
assert test_clips.groupby("id").size().eq(NUM_TEST_CLIPS).all()

print("Test videos:", len(test_data))
print("Test clips:", len(test_clips))
print("Test batches:", len(test_loader))


Test videos: 1344
Test clips: 4032
Test batches: 2016


## 10. Run Inference on Test Data

Calculate the collision probability for each test clip and store the clip-level predictions for aggregation into one final prediction per video.


In [20]:
model.eval()
test_predictions = []

with torch.inference_mode():
    for videos, labels, video_ids, horizons in tqdm(
        test_loader,
        desc="Test inference"
    ):
        videos = videos.to(device, non_blocking=True)
        logits = model(videos)
        probabilities = torch.softmax(logits.float(), dim=1)[:, 1]

        for video_id, score in zip(video_ids, probabilities.cpu().numpy()):
            test_predictions.append({
                "id": normalize_id(video_id),
                "score": float(score)
            })

test_predictions = pd.DataFrame(test_predictions)

assert len(test_predictions) == len(test_clips)
assert test_predictions["id"].nunique() == len(test_data)
assert np.isfinite(test_predictions["score"]).all()
assert test_predictions["score"].between(0, 1).all()

test_predictions.to_csv(
    WORK_ROOT / "test_clip_predictions.csv",
    index=False
)

print("Predicted clips:", len(test_predictions))
print("Unique test videos:", test_predictions["id"].nunique())
print("Probability range:", test_predictions["score"].min(), test_predictions["score"].max())


Test inference:   0%|          | 0/2016 [00:00<?, ?it/s]

Predicted clips: 4032
Unique test videos: 1344
Probability range: 0.14211927354335785 0.9457403421401978


## 11. Create and Validate the Submission File

For each video, use the highest prediction score among its clips as the final video-level score. Match the video IDs with the sample submission file, create the output CSV, and validate its structure and values.


In [21]:
video_predictions = (
    test_predictions
    .groupby("id", as_index=False)["score"]
    .max()
    .rename(columns={"score": "target"})
)

submission = sample_submission[["id"]].merge(
    video_predictions,
    on="id",
    how="left",
    validate="one_to_one"
)

submission = submission[sample_submission.columns.tolist()]
submission["target"] = submission["target"].astype(float)

assert len(submission) == len(sample_submission)
assert submission["id"].is_unique
assert submission["target"].notna().all()
assert submission["target"].between(0, 1).all()
assert set(submission["id"]) == set(sample_submission["id"])

submission_path = WORK_ROOT / "submission.csv"
submission.to_csv(submission_path, index=False)

print("Submission saved:", submission_path)
print("Shape:", submission.shape)
print("\nFirst rows:")
print(submission.head(10).to_string(index=False))
print("\nMissing values:")
print(submission.isna().sum())
print("\nSubmission is ready.")


Submission saved: /kaggle/working/submission.csv
Shape: (1344, 2)

First rows:
   id   target
00204 0.488887
00030 0.356846
00146 0.470695
00020 0.731022
00511 0.717965
00261 0.909628
00346 0.751284
00545 0.791946
00492 0.444037
00299 0.448209

Missing values:
id        0
target    0
dtype: int64

Submission is ready.
